# Efficient de-reservation in matching markets

This notebook reproduces the figures and tables for the de-reservation study.
The paper profile contains 3,200 paired synthetic markets across 32 settings. Six methods
per setting yield 19,200 recorded outcomes. The study compares vertical reservations
in a stylized Indian market and reservations with overlapping eligibility.

Run the notebook to read the bundled observations and regenerate the reports. See the
[replication guide](../README.md) for setup, algorithm definitions, and the data dictionary.

## 1. Setup

Install the requirements from the repository root and open this notebook in the same
Python environment.

    python -m pip install -r dereservation_study/requirements.txt
    jupyter lab dereservation_study/experiments.ipynb

The paper profile reads the bundled archive. The smoke and quick profiles generate
smaller studies on a first run and reuse their saved results on subsequent runs.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'dereservation_study' / 'generation.py').is_file()
             and (p / 'src' / 'instance_generation.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from inside the repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from dereservation_study.algorithms import algorithm_suite
from dereservation_study.analysis import (
    SWEEP_METRICS, audit_results, benchmark_weighted_engines, plot_sweep,
    runtime_statistics, save_figure,
)
from dereservation_study.examples import bt_avoidable_vacancies, compliance_tradeoff
from dereservation_study.experiments import METRICS, paired_differences, proposed_design, summarize
from dereservation_study.generation import generate_overlapping_market
from dereservation_study.metrics import evaluate_outcome
from dereservation_study.policies import compliance_frontier
from dereservation_study.reporting import PAPER_ARCHIVE, load_or_run_study, load_runtime_cache

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
PROFILE = 'paper'  # 'paper', 'quick', or 'smoke'
BASE_SEED = 271828
RESUME = True
PLOT_ALL_SWEEPS = True
RUN_RUNTIME_SCALING = True

OUTPUT_DIR = ROOT / 'dereservation_study' / 'results' / f'notebook_{PROFILE}'
REPORT_DIR = OUTPUT_DIR / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR = PAPER_ARCHIVE if PROFILE == 'paper' and BASE_SEED == 271828 else None

scenarios, repetitions = proposed_design(PROFILE)
suites = {regime: algorithm_suite(regime) for regime in ('indian', 'overlapping')}

def saved_file(name):
    local = OUTPUT_DIR / name
    if RESUME and not local.is_file() and ARCHIVE_DIR is not None:
        return ARCHIVE_DIR / name
    return local

print(f'{PROFILE}: {len(scenarios)} settings, {repetitions} paired draws per setting')
display(pd.DataFrame([{'regime': s.regime, 'factor': s.axis, 'value': s.value}
                     for s in scenarios]).groupby(['regime', 'factor']).value.agg(list))

### Mechanisms and design

- **Indian vertical reservations.** Fixed-quota DA fills open seats first, then
  reserved seats. Multi-run DA converts unused OBC quotas between complete DA
  runs. BT handles conversions within the school's choice step. SC and ST seats
  remain restricted.
- **Overlapping eligibility.** DA-open uses school priorities and aggregate
  capacity. Smart-Hard-Reserves checks whether a cohort can fit the original
  open and reserved seats. Soft-Reserves applies demographic floors at
  rejection. One student can count toward several floors.
- **Both settings.** MD maximizes local headcount then category-priority
  claims. RP puts original reserved-seat use between these two objectives.
  Merit maximizes headcount then the admitted cohort's total merit score.

Each draw gives all mechanisms the same market. The paper varies OBC supply,
type overlap, and one additional factor at a time. The full design is in the
archived manifest.

## 2. Reproduce the manuscript's avoidable-vacancy example

Example 2.1 has eight students and eight positions. BT and multi-run DA admit six;
MD can move admitted SC/ST students to their dedicated positions and admit all eight.
Original physical seat labels are retained throughout the comparison.

In [ ]:
example_path = saved_file('paper_example_2_1.csv')
if RESUME and example_path.exists():
    example_table = pd.read_csv(example_path)
else:
    toy = bt_avoidable_vacancies()
    example_rows = []
    for algorithm in algorithm_suite('indian'):
        outcome = algorithm.run(toy)
        metrics = evaluate_outcome(toy, outcome, algorithm, runtime=0)
        example_rows.append({'algorithm': algorithm.name,
                             'matched': metrics['num_matches'],
                             'original_reserved': metrics['original_reserved_seats'],
                             'ED_seats': metrics['dereserved_seats'],
                             'choice_blocks': metrics['blocking_pairs'],
                             'assignment': str(outcome.assignments[0])})
    example_table = pd.DataFrame(example_rows)
    example_table.to_csv(OUTPUT_DIR / 'paper_example_2_1.csv', index=False)
display(example_table)
assert example_table.set_index('algorithm').loc['BT', 'matched'] == 6
assert example_table.set_index('algorithm').loc['MD', 'matched'] == 8

## 3. Load the paired synthetic study

We load saved observations after checking the design, seeds, replication count,
and algorithm definitions. A fresh clone can use the compressed paper archive.
The outcome table keeps one row per setting, seed, and algorithm.

An original reserved position filled by an originally eligible student contributes
to $D$. A reserved position filled by an additional eligible student contributes
to $T$. Together, $D+T$ counts the occupied original reserved positions. BT and
multi-run DA also record which OBC quotas were converted administratively.

In [ ]:
study = load_or_run_study(
    scenarios, repetitions, BASE_SEED, suites, OUTPUT_DIR / 'main',
    resume=RESUME, archive_dir=ARCHIVE_DIR,
)
display(pd.DataFrame([audit_results(study)]))
feasible = study.outcomes.loc[study.outcomes.eligibility_feasible]
outcome_metrics = [metric for metric in METRICS if metric != 'runtime']

### Domain and label checks before interpreting comparisons

The original soft-bound code can return an unsupported cohort when new transfer
permissions are restricted. Such outputs remain in the raw file, but default
summaries and paired differences exclude them. Undefined seat metrics are missing,
not zero. Full transfer access in the default overlap study makes these benchmark
cohorts implementable.

Earlier backends report placements only. Their `posthoc_MD_support` seat labels
are a deterministic reporting convention. `original_reserved_seats_max_for_cohort`
also measures the greatest D for the same students; this reveals label sensitivity.
The general benchmarks use their existing school priorities and soft-bound
interpretations; they are not relabeled as exact implementations of the new model.

In [ ]:
domain = (study.outcomes.groupby(['regime', 'axis', 'value', 'algorithm'], as_index=False)
          .agg(runs=('eligibility_feasible', 'size'),
               feasible_runs=('eligibility_feasible', 'sum')))
domain['unsupported_runs'] = domain.runs - domain.feasible_runs
display(domain.loc[domain.unsupported_runs.gt(0)])
domain.to_csv(REPORT_DIR / 'domain_checks.csv', index=False)

summary = summarize(study.outcomes, outcome_metrics)
summary.to_csv(REPORT_DIR / 'metric_summary.csv', index=False)
runtime_statistics(feasible, ['regime', 'scenario', 'axis', 'value', 'algorithm']).to_csv(
    REPORT_DIR / 'algorithm_runtime.csv', index=False,
)

## 4. Welfare, seat use, envy, waste, and representation

Algorithms share each metric's axes. Wider strokes and distinct dash patterns
show overlaid curves. Symbols between measured settings identify exactly
coincident segments. Values at measured settings and error bars keep their
original coordinates. Gaps indicate settings with no feasible observations.

Plots show means and approximate 95% confidence intervals across draws.
Runtime shows the median and interquartile range. The raw timing series
include isolated long observations.

The average rank includes students who remain unmatched at rank one plus
their list length. Protected shares count each student once, including
students eligible in several categories.

In [ ]:
main_sweeps = {('indian', 'obc_population'), ('overlapping', 'overlap')}
sweeps = study.outcomes[['regime', 'axis']].drop_duplicates().itertuples(index=False, name=None)
for regime, axis in sweeps:
    if not PLOT_ALL_SWEEPS and (regime, axis) not in main_sweeps:
        continue
    for metrics in SWEEP_METRICS.values():
        fig = plot_sweep(study.outcomes, regime, axis, metrics, output_dir=FIGURE_DIR)
        if (regime, axis) in main_sweeps:
            display(fig)
        plt.close(fig)

### Three blocking diagnostics

`blocking_pairs` uses the paper's test: a student prefers the school and its
announced choice accepts her from its current cohort plus that student. Category
exchanges are allowed. `choice_waste_pairs` increases headcount; `choice_envy_pairs`
is the replacement case. These two counts partition policy blocking.

`category_envy_pairs` and `category_waste_pairs` inspect the reported category
assignment without exchanges: a lower-priority occupant or an eligible vacancy.
They may overlap; their union is saved separately. `merit_*` diagnostics ignore
reservation protections. All count student-school edges, not student-student
triples or numbers of vacant seats. Use `empty_seats` for the latter.

Multi-run DA is tested under frozen final quotas. A zero policy-blocking count
does not imply zero ordinary merit envy. The legacy overlapping soft-bound fallback
is checked empirically, without assuming a general substitutes theorem.

In [ ]:
stability_columns = ['blocking_pairs', 'choice_envy_pairs', 'choice_waste_pairs',
                     'category_envy_pairs', 'category_waste_pairs', 'category_blocking_pairs',
                     'merit_envy_pairs', 'merit_waste_pairs', 'merit_blocking_pairs']
display(feasible.groupby(['regime', 'algorithm'])[stability_columns].mean().round(3))
assert (study.outcomes.blocking_pairs ==
        study.outcomes.choice_envy_pairs + study.outcomes.choice_waste_pairs).all()

## 5. Paired comparisons and student welfare relative to BT

Differences are algorithm minus reference on the same instance. Negative rank
differences and positive matching differences favor the algorithm. Uncertainty
is computed on paired differences, not by combining separate algorithm error bars.
Student-level welfare is a separate statistic: students better, equal, or worse
relative to BT in each Indian market. The Indian MD-versus-BT comparison is an
empirical consistency check in the theorem's domain, not a guarantee for other
weighted policies or general overlapping markets.

In [ ]:
paired_bt = paired_differences(study.outcomes, 'BT', outcome_metrics)
paired_soft = paired_differences(study.outcomes, 'Soft-Reserves', outcome_metrics)
paired_bt.to_csv(REPORT_DIR / 'paired_vs_BT.csv', index=False)
paired_soft.to_csv(REPORT_DIR / 'paired_vs_Soft-Reserves.csv', index=False)
display(paired_bt[(paired_bt.algorithm == 'MD') &
                  paired_bt.metric.isin(['average_rank', 'num_matches', 'original_reserved_seats'])]
        [['scenario', 'metric', 'n', 'mean', 'ci95_low', 'ci95_high']])

welfare = study.welfare[(study.welfare.regime == 'indian') &
                        (study.welfare.algorithm == 'MD') &
                        study.welfare.eligibility_feasible]
display(welfare[['scenario', 'seed', 'students_better', 'students_equal',
                 'students_worse', 'mean_rank_gain']])
assert (welfare.students_worse == 0).all()

by_supply = welfare[welfare.axis == 'obc_population'].groupby('value')[['share_better', 'share_worse']].mean()
by_supply['share_equal'] = 1 - by_supply.share_better - by_supply.share_worse
fig, ax = plt.subplots(figsize=(7, 4))
by_supply[['share_better', 'share_equal', 'share_worse']].plot.bar(stacked=True, ax=ax, rot=0)
ax.set(xlabel='OBC population share', ylabel='Share of students',
       title='MD student ranks relative to BT (paired synthetic instances)')
ax.legend(['Better', 'Equal', 'Worse'], loc='upper right')
save_figure(fig, FIGURE_DIR, 'indian_md_vs_bt_student_welfare')
display(fig)
plt.close(fig)

### Per-group representation

Per-type populations and counts can overlap. ANY_PROTECTED counts the union
and UNPROTECTED counts its complement. The table selects the low OBC supply
setting and an overlapping setting for the chosen profile.

In [ ]:
selected = feasible[
    ((feasible.axis == 'obc_population') & (feasible.value == 0.02)) |
    ((feasible.axis == 'overlap') & (feasible.value == {'paper': 0.5, 'quick': 0.5, 'smoke': 0.8}[PROFILE]))
][['instance_id', 'algorithm']]

group_table = (study.groups.merge(selected, on=['instance_id', 'algorithm'], validate='many_to_one')
               .groupby(['scenario', 'algorithm', 'group'], as_index=False)
               .agg(mean_population=('group_population', 'mean'),
                    mean_matched=('group_matched', 'mean'),
                    admission_rate=('group_admission_rate', 'mean'),
                    share_of_matched=('group_share_matched', 'mean'),
                    average_rank=('group_average_rank', 'mean')))
display(group_table[group_table.group == 'ANY_PROTECTED'])
group_table.to_csv(REPORT_DIR / 'group_summary_selected_scenarios.csv', index=False)

## 6. Exact local original-seat compliance frontier

Fix one institution and one applicant pool. For each feasible headcount k,
D*(k) is the greatest possible original reserved-seat compliance. Example 6.4
admits three students with D=1, or two with D=2. This is a local descriptive
frontier, separate from market-wide stable matching outcomes. Saved frontier
tables are reused without recomputing flow solutions.

In [ ]:
example_frontier_path = saved_file('example_6_4_local_frontier.csv')
generated_frontier_path = saved_file('generated_local_frontier.csv')
if RESUME and example_frontier_path.exists():
    frontier = pd.read_csv(example_frontier_path)
else:
    frontier = pd.DataFrame(compliance_frontier(compliance_tradeoff(), 0)).drop(columns='assignment')
display(frontier)
if not RESUME or not example_frontier_path.exists():
    frontier.to_csv(OUTPUT_DIR / 'example_6_4_local_frontier.csv', index=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(frontier.headcount, frontier.original_reserved_seats, 'o-', label='Exact D*(k)')
ax.set(xlabel='Admissions at the fixed institution', ylabel='Original reserved seats filled by E0',
       title='Manuscript Example 6.4: local compliance frontier')
save_figure(fig, FIGURE_DIR, 'example_6_4_frontier')
display(fig)
plt.close(fig)
if RESUME and generated_frontier_path.exists():
    generated_frontier = pd.read_csv(generated_frontier_path)
else:
    local_market = generate_overlapping_market(seed=BASE_SEED, num_students=24, num_institutions=3,
                                               reserve_fraction=.8, transferable_fraction=.5,
                                               transfer_access=.5, priority_noise=.5)
    generated_frontier = pd.DataFrame(compliance_frontier(local_market, 0)).drop(columns='assignment')
display(generated_frontier)
if not RESUME or not generated_frontier_path.exists():
    generated_frontier.to_csv(OUTPUT_DIR / 'generated_local_frontier.csv', index=False)

## 7. Runtime of the two exact MD implementations

The benchmark checks that incremental alternating-path updates and recomputed
minimum-cost flows return identical assignments. Plot points are medians and
error bars show the middle half of the recorded times. Both axes are logarithmic.
The paper profile comes with 20 paired trials at each size.

In [ ]:
if RUN_RUNTIME_SCALING:
    sizes = {'smoke': (24, 48, 96), 'quick': (100, 200, 400, 800),
             'paper': (100, 250, 500, 1000, 2000)}[PROFILE]
    runtime_reps = {'smoke': 2, 'quick': 5, 'paper': 20}[PROFILE]
    schools = {'smoke': 4, 'quick': 8, 'paper': 20}[PROFILE]
    runtime_path = saved_file('runtime_engines.csv')
    if RESUME and runtime_path.is_file():
        runtime_data = load_runtime_cache(runtime_path, sizes, runtime_reps, schools)
    else:
        runtime_data = benchmark_weighted_engines(sizes, runtime_reps, num_institutions=schools)
        runtime_data.to_csv(OUTPUT_DIR / 'runtime_engines.csv', index=False)

    runtime_summary = runtime_statistics(runtime_data, ['engine', 'num_students'])
    display(runtime_summary)
    runtime_summary.to_csv(REPORT_DIR / 'runtime_summary.csv', index=False)

    fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
    styles = {'flow': ('Recomputed flow', '#2865a8', '-', 's', 3.0),
              'incremental': ('Alternating path', '#007f66', '--', 'o', 1.7)}
    for engine, group in runtime_summary.groupby('engine'):
        label, color, linestyle, marker, width = styles[engine]
        ax.errorbar(group.num_students, group['median'],
                    yerr=[group['median'] - group.q1, group.q3 - group['median']],
                    color=color, linestyle=linestyle, linewidth=width, marker=marker,
                    markersize=5, capsize=3, label=label)
    ax.set(xlabel='Students', ylabel='Median DA runtime (seconds)', xscale='log', yscale='log')
    ax.set_xticks(sizes, labels=[str(size) for size in sizes])
    ax.legend(frameon=False)
    save_figure(fig, FIGURE_DIR, 'runtime_engines')
    display(fig)
    plt.close(fig)

## 8. Reports

Each profile writes its tables and figures to its local reports directory.
The README describes the archive format and the commands for reproducing the
analysis.

In [ ]:
print('Report directory:', REPORT_DIR)
display(pd.DataFrame({'table': sorted(p.name for p in REPORT_DIR.glob('*.csv'))}))